# Followchon

## Variables

- Quantitative
    - Continue
        - Température 
- Qualitative
    - Ordinal
        - Heure
    - Nominal
        - Class (Noisette ou Stitch)
        - Zone (Clapier, Cachette, Fontaine, ...)
- Autres
    - Date
        - Date du jour
        - Date de la capture
    - Coordonnées
        - Coordonnée du chon (en valeur normal [0:1] )

## Récupération des détections

In [9]:
from django.db import connection
from detections.models import Detection
from configuration.models import Zone, Family
from datetime import datetime
from plotly.graph_objects import FigureWidget
from IPython.display import display
from ipywidgets import HBox, VBox, Box, fixed, interactive_output

import os
import csv
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
import plotly.graph_objects as go

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

In [107]:
detections_query =  'SELECT d.id, ' + \
                    'STRFTIME("%Y-%m-%d", c.date) AS date, ' + \
                    'STRFTIME("%Y-%m-%d %H:%M:%S", c.date) AS datetime, ' + \
                    'STRFTIME("%H", c.date) AS hour, ' + \
                    'STRFTIME("%Y%m%d%H", c.date) AS datehour_key, ' +  \
                       'z.id AS zone_id, ' +  \
                       'z.name AS zone_name, ' +  \
                       'f.[index] AS class_index, ' +  \
                       'f.name AS class_name, ' +  \
                       'd.center_x AS center_x_norm, ' +  \
                       'd.center_y AS center_y_norm ' +  \
                  'FROM detections_detection d ' +  \
                       'LEFT JOIN ' +  \
                       'detections_capture c ON d.capture_id = c.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_family f ON d.family_id = f.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_zone z ON d.zone_id = z.id ' +  \
                 'WHERE c.status == "archived" AND  ' +  \
                       'c.source == "vision" AND  ' +  \
                       '(f.[index] == 1 OR  ' +  \
                        'f.[index] == 2)  ' +  \
                 'ORDER BY c.date ASC '
                        
detections = Detection.objects.raw(detections_query)

zones_query = 'SELECT z.id, z.id AS zone_id, z.name AS zone_name FROM configuration_zone z ORDER BY z.id ASC'
zones = Zone.objects.raw(zones_query)

classes_query = 'SELECT f.id, f.[index] AS class_index, f.name AS class_name FROM configuration_family f ORDER BY f.id ASC'
classes = Family.objects.raw(classes_query)

def save_rows(rows, query, path): 
    columns = list()
    
    with connection.cursor() as cursor:
        cursor.execute(query)
        columns = [col[0] for col in cursor.description]
    
    with open(path, 'w+', newline='') as file:
        writer = csv.writer(file)

        writer.writerow(columns)
        
        for obj in rows:
            row = list()
            
            for col in columns:
                row.append(obj.__dict__[col])
                
            writer.writerow(row)
    
        print(f"{len(rows)} lignes exportées")

save_rows(detections, detections_query, 'data/detections.csv')
save_rows(zones, zones_query, 'data/zones.csv')
save_rows(classes, classes_query, 'data/classes.csv')

53474 lignes exportées
14 lignes exportées
5 lignes exportées


In [122]:
df_detections = pd.read_csv('data/detections.csv')
df_zones = pd.read_csv('data/zones.csv')
df_classes = pd.read_csv('data/classes.csv')

df_detections.sample(5)

,id,date,datetime,hour,datehour_key,zone_id,zone_name,class_index,class_name,center_x_norm,center_y_norm
48011,178224,2024-11-05,2024-11-05 09:07:01,9,2024110509,1.0,Clapier,1,Noisette,0.393130,0.216804
17638,55298,2024-09-11,2024-09-11 19:41:00,19,2024091119,18.0,Maison,2,Stitch,0.474201,0.662995
41352,164700,2024-10-23,2024-10-23 17:14:00,17,2024102317,NaN,NaN,2,Stitch,0.556152,0.382812
42964,167350,2024-10-26,2024-10-26 18:55:01,18,2024102618,2.0,Cachette,1,Noisette,0.643826,0.182944
31952,110977,2024-09-30,2024-09-30 08:37:00,8,2024093008,2.0,Cachette,2,Stitch,0.628426,0.173175


## Jointure avec les données météorologique

In [109]:
df_meteo = pd.read_csv('data/H_69_latest-2023-2024.csv', sep=";")
df_meteo.sample(5)

,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJHH,RR1,QRR1,DRR1,QDRR1,...,INS,QINS,INS2,QINS2,TLAGON,QTLAGON,TVEGETAUX,QTVEGETAUX,ECOULEMENT,QECOULEMENT
75652,69114001,LIERGUES_SAPC,45.976500,4.650333,290,2024040417,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31520,69028001,BRINDAS,45.713333,4.693167,317,2024100204,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
161513,69196001,ST-DIDIER-BEAUJ,46.164167,4.566333,345,2024110212,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
191659,69204002,ST-GENIS-LAVAL,45.694667,4.782333,290,2024080406,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
286553,69299001,LYON-ST EXUPERY,45.726500,5.077833,235,2024051104,0.0,1.0,0.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [110]:
detections_columns = ["datetime", "date", "hour", "zone_name", "zone_id", "class_name", "class_index"]
meteo_poste_columns = ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI']
meteo_temp_columns = ['AAAAMMJJHH', 'T']

mask_date = df_meteo["AAAAMMJJHH"] > df_detections['datehour_key'][0]
mask_lat_lon = df_meteo['LAT'].between(45.7, 45.8) & df_meteo['LON'].between(4.7, 5.1)

df_meteo_sorted = df_meteo.loc[mask_date & mask_lat_lon, meteo_poste_columns + meteo_temp_columns].sort_values(by=['AAAAMMJJHH', 'ALTI'])
df_meteo_reduced = df_meteo_sorted[meteo_temp_columns + meteo_poste_columns[-1:]].drop_duplicates(['AAAAMMJJHH'])

df_detections_t = pd.merge(df_detections, df_meteo_reduced, left_on="datehour_key", right_on="AAAAMMJJHH", how="left")[detections_columns + meteo_temp_columns[1:]]
df_detections_t = df_detections_t.rename(columns={'zone_name': 'zone', 'class_name' : 'class'})

df_detections_t.to_csv('data/df_detections_t.csv', index=False)

df_detections_t.sample(5)

,datetime,date,hour,zone,zone_id,class,class_index,T
42788,2024-10-26 18:18:00,2024-10-26,18,Cachette,2.0,Noisette,1,16.1
21601,2024-09-15 19:15:01,2024-09-15,19,Loft,14.0,Stitch,2,15.5
49720,2024-11-09 16:45:01,2024-11-09,16,Passerelle,13.0,Noisette,1,NaN
9808,2024-08-30 20:49:01,2024-08-30,20,Cachette,2.0,Stitch,2,25.9
8473,2024-08-26 19:42:38,2024-08-26,19,Centre haut,9.0,Noisette,1,21.5


In [112]:
df_detections_t = pd.read_csv('data/df_detections_t.csv')

df_detections_t['date'] = pd.to_datetime(df_detections_t['datetime'])
df_detections_t['datetime'] = pd.to_datetime(df_detections_t['datetime'])

df_detections_t['zone'] = (df_detections_t['zone'].astype(str)).str.replace('nan', '')
df_detections_t['class'] = df_detections_t['class'].astype(str)

df_detections_t['T'] = pd.to_numeric(df_detections_t['T'])
df_detections_t['zone_id'] = pd.to_numeric(df_detections_t['zone_id'])
df_detections_t['class_index'] = pd.to_numeric(df_detections_t['class_index'])
df_detections_t['hour'] = pd.to_numeric(df_detections_t['hour'])

df_detections_t.sample(5)

,datetime,date,hour,zone,zone_id,class,class_index,T
9250,2024-08-29 09:38:46,2024-08-29 09:38:46,9,Clapier,1.0,Stitch,2,26.7
18374,2024-09-12 10:26:00,2024-09-12 10:26:00,10,Foin,16.0,Noisette,1,13.6
33686,2024-10-03 08:24:01,2024-10-03 08:24:01,8,Passerelle,13.0,Stitch,2,13.6
4070,2024-08-13 20:31:09,2024-08-13 20:31:09,20,Clapier,1.0,Stitch,2,26.6
8518,2024-08-26 20:38:30,2024-08-26 20:38:30,20,Cachette,2.0,Noisette,1,20.9


In [170]:
df_temp_by_date = df_detections_t.loc[(~df_detections_t['T'].isnull()), ['datetime', 'T']]\
    .groupby('datetime').mean()

df_mean_temp_by_date = df_temp_by_date.resample('D').mean().reset_index().rename(columns={'datetime' : 'date'})
df_mean_temp_by_date = df_mean_temp_by_date[~df_mean_temp_by_date['T'].isnull()]

df_mean_temp_by_date.to_csv('data/df_mean_temp_by_date.csv', index=False)

df_temp_by_datehour = df_detections_t.loc[(~df_detections_t['T'].isnull()), ['datetime', 'T']]\
    .groupby('datetime').mean()

df_mean_temp_by_datehour = df_temp_by_datehour.resample('h').mean().reset_index().rename(columns={'datetime' : 'datehour'})
df_mean_temp_by_datehour = df_mean_temp_by_datehour[~df_mean_temp_by_datehour['T'].isnull()]

df_mean_temp_by_datehour.to_csv('data/df_mean_temp_by_datehour.csv', index=False)

df_mean_temp_by_datehour.sample(5)

,datehour,T
1153,2024-09-07 15:00:00,29.7
355,2024-08-05 09:00:00,25.2
867,2024-08-26 17:00:00,23.8
2324,2024-10-26 10:00:00,16.3
1442,2024-09-19 16:00:00,24.0


## Traitements des données

In [225]:
zones_all = ['Clapier', 'Cachette', 'Fontaine', 'Passerelle', 'Loft', 'Foin']
classes_all = ['Noisette', 'Stitch']
dates_all = df_detections_t['date'].unique()

hour_begin = 9
hour_end = 19
hours_all = list(range(hour_begin, hour_end + 1))

In [206]:
def calc_duration_by_zone(df, class_name):
    df_filterd = df.loc[df["class"] == class_name]
    
    set_duration_by_zone_and_day = dict()
    day_previous = None
    zone_previous = None
    datetime_zone_enter = None

    for i in df_filterd.index:
        day_current = df_filterd['datetime'][i].strftime('%Y-%m-%d')
        datetime_current = df_filterd['datetime'][i]
        zone_current = df_filterd['zone'][i]

        if day_previous is None or day_previous != day_current:
            day_previous = day_current
            zone_previous = None
            datetime_zone_enter = None
            set_duration_by_zone_and_day[day_previous] = {}

        if zone_previous is not None and datetime_zone_enter is not None and zone_previous != zone_current:
            zone_duration = datetime_current - datetime_zone_enter

            set_duration_by_zone_and_day[day_previous][zone_previous] = \
                zone_duration if zone_previous not in set_duration_by_zone_and_day[day_previous] \
                else set_duration_by_zone_and_day[day_previous][zone_previous] + zone_duration

        if zone_previous is None or zone_previous != zone_current:
            zone_previous = zone_current
            datetime_zone_enter = datetime_current

    df_duration_by_zone = pd.concat(
        { k: pd.DataFrame.from_dict(v, orient='index') for k, v in set_duration_by_zone_and_day.items() },
        axis=0,
    ).reset_index().rename(columns={'level_0': 'date', 'level_1': 'zone', 0: 'duration'})

    df_duration_by_zone['class'] = class_name
    df_duration_by_zone['date'] = pd.to_datetime(df_duration_by_zone['date'])
    df_duration_by_zone['duration'] = pd.to_timedelta(df_duration_by_zone['duration'], unit='s').dt.total_seconds() / 60
    
    df_duration_by_zone_merged = pd.merge(df_duration_by_zone, df_mean_temp_by_date, left_on="date", right_on="date", how="left")
    
    df_duration_by_zone_merged = pd.merge(df_duration_by_zone_merged, df_zones, left_on="zone", right_on="zone_name", how="left")
    df_duration_by_zone_merged = df_duration_by_zone_merged.drop(columns=['id', 'zone_name'])
    
    df_duration_by_zone_merged = pd.merge(df_duration_by_zone_merged, df_classes, left_on="class", right_on="class_name", how="left")
    df_duration_by_zone_merged = df_duration_by_zone_merged.drop(columns=['id', 'class_name'])
    
    df_duration_by_zone_filtered = df_duration_by_zone_merged[
        ~df_duration_by_zone_merged['zone'].eq('') 
        | ~df_duration_by_zone_merged['T'].isnull()
    ]

    return df_duration_by_zone_filtered

In [266]:
df_n_duration = calc_duration_by_zone(df_detections_t, classes_all[0])
df_s_duration = calc_duration_by_zone(df_detections_t, classes_all[1])

df_duration = pd.concat([df_n_duration ,df_s_duration], ignore_index=True)
                              
df_duration.to_csv('./data/df_duration.csv', index=False)

df_duration

,date,zone,duration,class,T,zone_id,class_index
0,2024-07-21,Clapier,18.300000,Noisette,25.210658,1.0,1
1,2024-07-21,,284.733333,Noisette,25.210658,NaN,1
2,2024-07-21,Cachette,141.700000,Noisette,25.210658,2.0,1
3,2024-07-21,Bas,8.466667,Noisette,25.210658,6.0,1
4,2024-07-22,,64.533333,Noisette,25.212821,NaN,1
...,...,...,...,...,...,...,...
1686,2024-11-18,Loft,1.016667,Stitch,NaN,14.0,2
1687,2024-11-19,Clapier,0.000000,Stitch,NaN,1.0,2
1688,2024-11-19,Cachette,13.000000,Stitch,NaN,2.0,2
1689,2024-11-19,Foin,10.000000,Stitch,NaN,16.0,2


In [267]:
df_duration_by_zone = df_duration.loc[:, ['zone', 'zone_id', 'class', 'class_index', 'duration']]\
    .groupby(['zone', 'zone_id', 'class', 'class_index'])\
    .mean('duration')\
    .reset_index()

df_duration_by_zone

,zone,zone_id,class,class_index,duration
0,Bas,6.0,Noisette,1,21.319792
1,Bas,6.0,Stitch,2,15.716026
2,Cachette,2.0,Noisette,1,223.564851
3,Cachette,2.0,Stitch,2,210.177723
4,Centre bas,10.0,Noisette,1,19.219271
5,Centre bas,10.0,Stitch,2,19.393889
6,Centre haut,9.0,Noisette,1,19.088384
7,Centre haut,9.0,Stitch,2,22.356061
8,Clapier,1.0,Noisette,1,153.072727
9,Clapier,1.0,Stitch,2,155.451852


In [221]:
def calc_occupation_by_zone(df, class_name):
    df_filterd = df.loc[df["class"] == class_name]
    
    set_duration_by_zone_and_hour_for_days = dict()
    day_previous = None
    zone_previous = None
    datetime_zone_enter = None

    for i in df.index:
        day_current = df['datetime'][i].strftime('%Y-%m-%d')
        datetime_current = df['datetime'][i]
        zone_current = df['zone'][i]

        if day_previous is None or day_previous != day_current:
            day_previous = day_current
            zone_previous = None
            datetime_zone_enter = None

        if zone_previous is not None and datetime_zone_enter is not None and zone_previous != zone_current:
            hour_enter = datetime_zone_enter.floor('h')
            hour_exit = datetime_current.floor('h')

            if day_previous not in set_duration_by_zone_and_hour_for_days:
                set_duration_by_zone_and_hour_for_days[day_previous] = {}

            if zone_previous not in set_duration_by_zone_and_hour_for_days[day_previous]:
                set_duration_by_zone_and_hour_for_days[day_previous][zone_previous] = {}

            hour_current = hour_enter
            while hour_current <= hour_exit:
                interval_begin = max(datetime_zone_enter, hour_current)
                interval_end = min(datetime_current, hour_current + pd.Timedelta(hours=1))

                interval = (interval_end - interval_begin).total_seconds() / 3600
                set_duration_by_zone_and_hour_for_days[day_previous][zone_previous][hour_current.hour] = \
                    interval if hour_current.hour not in set_duration_by_zone_and_hour_for_days[day_previous][zone_previous] \
                    else set_duration_by_zone_and_hour_for_days[day_previous][zone_previous][hour_current.hour] + interval

                hour_current += pd.Timedelta(hours=1)

        if zone_previous is None or zone_previous != zone_current:
            zone_previous = zone_current
            datetime_zone_enter = datetime_current

    df_occupation_by_zone = pd.DataFrame([
        {'date': date, 'datehour' : f"{date} {hour}:00:00", 'zone': zone, 'hour': hour, 'occupation': occupation}
        for date, by_date in set_duration_by_zone_and_hour_for_days.items()
        for zone, hours in by_date.items()
        for hour, occupation in hours.items()
    ])

    df_occupation_by_zone['class'] = class_name
    df_occupation_by_zone['date'] = pd.to_datetime(df_occupation_by_zone['date'])
    df_occupation_by_zone['datehour'] = pd.to_datetime(df_occupation_by_zone['datehour'])
    df_occupation_by_zone['hour'] = pd.to_numeric(df_occupation_by_zone['hour'])
    
    df_occupation_by_zone_merged = pd.merge(df_occupation_by_zone, df_mean_temp_by_datehour, left_on="datehour", right_on="datehour", how="left")
    
    df_occupation_by_zone_merged = pd.merge(df_occupation_by_zone_merged, df_zones, left_on="zone", right_on="zone_name", how="left")
    df_occupation_by_zone_merged = df_occupation_by_zone_merged.drop(columns=['id', 'zone_name'])
    
    df_occupation_by_zone_merged = pd.merge(df_occupation_by_zone_merged, df_classes, left_on="class", right_on="class_name", how="left")
    df_occupation_by_zone_merged = df_occupation_by_zone_merged.drop(columns=['id', 'class_name'])
    
    df_occupation_by_zone_filtered = df_occupation_by_zone_merged[
        ~df_occupation_by_zone_merged['zone'].eq('') 
        | ~df_occupation_by_zone_merged['T'].isnull()
        | df_occupation_by_zone_merged['hour'].between(hour_begin, hour_end)
    ]

    return df_occupation_by_zone_filtered

In [222]:
df_n_occupation = calc_occupation_by_zone(df_detections_t, classes_all[0])
df_s_occupation = calc_occupation_by_zone(df_detections_t, classes_all[1])

df_occupation = pd.concat([df_n_occupation ,df_s_occupation], ignore_index=True)
                              
df_occupation.to_csv('./data/df_occupation.csv', index=False)

df_occupation

,date,datehour,zone,hour,occupation,class,T,zone_id,class_index
0,2024-07-21,2024-07-21 13:00:00,Clapier,13,0.018333,Noisette,NaN,1.0,1
1,2024-07-21,2024-07-21 15:00:00,Clapier,15,0.018056,Noisette,26.6,1.0,1
2,2024-07-21,2024-07-21 16:00:00,Clapier,16,0.003333,Noisette,25.5,1.0,1
3,2024-07-21,2024-07-21 17:00:00,Clapier,17,0.001111,Noisette,25.1,1.0,1
4,2024-07-21,2024-07-21 18:00:00,Clapier,18,0.005278,Noisette,25.1,1.0,1
...,...,...,...,...,...,...,...,...,...
10747,2024-11-19,2024-11-19 09:00:00,Cachette,9,0.333333,Stitch,NaN,2.0,2
10748,2024-11-19,2024-11-19 09:00:00,Passerelle,9,0.000000,Stitch,NaN,13.0,2
10749,2024-11-19,2024-11-19 09:00:00,Clapier,9,0.181389,Stitch,NaN,1.0,2
10750,2024-11-19,2024-11-19 09:00:00,Foin,9,0.166667,Stitch,NaN,16.0,2


In [264]:
df_occupation_by_hour = df_occupation.loc[:, ['hour', 'class', 'class_index', 'zone', 'zone_id', 'occupation']]\
    .groupby(['hour', 'class', 'class_index', 'zone', 'zone_id'])\
    .mean('occupation')\
    .reset_index()

df_occupation_by_hour

,hour,class,class_index,zone,zone_id,occupation
0,0,Noisette,1,Bas,6.0,0.011944
1,0,Noisette,1,Cachette,2.0,0.425556
2,0,Noisette,1,Centre bas,10.0,0.198056
3,0,Noisette,1,Centre haut,9.0,0.068056
4,0,Noisette,1,Clapier,1.0,0.081944
...,...,...,...,...,...,...
487,23,Stitch,2,Fontaine,5.0,0.047222
488,23,Stitch,2,Loft,14.0,0.000833
489,23,Stitch,2,Passerelle,13.0,0.000000
490,23,Stitch,2,Piscine,4.0,0.075556


In [309]:
df_occupation_by_zone = df_occupation.loc[:, ['zone', 'zone_id', 'class', 'class_index', 'occupation']]\
    .groupby(['zone', 'zone_id', 'class', 'class_index'])\
    .mean('occupation')\
    .reset_index()

df_occupation_by_zone

,zone,zone_id,class,class_index,occupation
0,Bas,6.0,Noisette,1,0.069830
1,Bas,6.0,Stitch,2,0.069830
2,Cachette,2.0,Noisette,1,0.418285
3,Cachette,2.0,Stitch,2,0.418285
4,Centre bas,10.0,Noisette,1,0.069221
5,Centre bas,10.0,Stitch,2,0.069221
6,Centre haut,9.0,Noisette,1,0.047355
7,Centre haut,9.0,Stitch,2,0.047355
8,Clapier,1.0,Noisette,1,0.295786
9,Clapier,1.0,Stitch,2,0.295786


## Fonctions d'affichage

In [302]:
box_layout = widgets.Layout(
    display='flex',
    flex_flow='row wrap',  # Définit l'orientation et l'autorise à passer à la ligne
    justify_content='space-around',  # Espacement autour des items
    align_items='center',  # Aligne les items au centre verticalement
    width='100%',  # Largeur du conteneur
)

def generate_widget_corr(df, title, columns, columns_to_remove, size=700):
    df_corr = df.loc[:, columns].corr()

    for c in columns:
        df_corr.loc[c, c] = 0
        
    for c1 in columns_to_remove:
        for c2 in columns_to_remove:
            df_corr.loc[c1, c2] = 0
    
    fig = px.imshow(
        df_corr,
        color_continuous_scale="rdbu",
        title=str(title),
        zmin=-1,
        zmax=1,
        text_auto=".2f",
    )

    fig.update_layout(width=size, height=size)
    fig.update_traces(textfont_size=16)

    return FigureWidget(fig)

def generate_widget_filter_zone(df, zone, columns, columns_to_remove, size=700):
    df_filtered = df.loc[df['zone'] == zone, columns]

    return generate_widget_corr(df_filtered, zone, columns, columns_to_remove, size)

def generate_widget_filter_class(df, class_name, columns, columns_to_remove, size=700):
    df_filtered = df.loc[df['class'] == class_name, columns]

    return generate_widget_corr(df_filtered, class_name, columns, columns_to_remove, size)

def generate_widget_filter_hour(df, hour, columns, columns_to_remove, size=700):
    df_filtered = df.loc[df['hour'] == hour, columns]

    return generate_widget_corr(df_filtered, hour, columns, columns_to_remove, size)

def generate_widget_all_classes(df, columns, columns_to_remove, size=700):
    widgets = list()
       
    for class_name in classes_all:
        widgets.append(
            generate_widget_filter_class(
                df, 
                class_name, 
                list(filter(lambda c: c != 'class' and c != 'class_index', columns)),
                columns_to_remove,
                size=size
            )
        )

    return widgets

def generate_widget_all_zones(df, columns, columns_to_remove, size=500):
    widgets = list()
       
    for zone in zones_all:
        widgets.append(
            generate_widget_filter_zone(
                df, 
                zone, 
                list(filter(lambda c: c != 'zone' and c != 'zone_id', columns)),
                columns_to_remove,
                size=size,
            )
        )

    return widgets

def generate_widget_all_hours(df, columns, columns_to_remove, size=500):
    widgets = list()
       
    for hour in hours_all:
        widgets.append(
            generate_widget_filter_hour(
                df, 
                hour, 
                list(filter(lambda c: c != 'hour', columns)),
                columns_to_remove,
                size=size,
            )
        )

    return widgets

def generate_box(children, layout):
    return Box(
        children=children, 
        layout=layout
    )

## Recherche de corrélation

### Par class

#### Sans regroupements

In [237]:
display(
    generate_box(
        children=generate_widget_all_classes(df_detections_t, ['datetime', 'hour', 'T', 'zone_id', 'class_index'], ['datetime', 'T'], 700),
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec durée par zone et jour

In [238]:
display(
    generate_box(
        children=generate_widget_all_classes(df_duration, ['date', 'duration', 'T', 'zone_id', 'class_index'], ['date', 'T'], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec durée par zone

In [269]:
display(
    generate_box(
        children=generate_widget_all_classes(df_duration_by_zone, ['duration', 'zone_id', 'class_index'], [], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par zone et par heure

In [242]:
display(
    generate_box(
        children=generate_widget_all_classes(df_occupation, ['date', 'hour', 'occupation', 'T', 'zone_id', 'class_index'], ['date', 'hour', 'T'], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par heure

In [273]:
display(
    generate_box(
        children=generate_widget_all_classes(df_occupation_by_hour, ['hour', 'occupation', 'zone_id', 'class_index'], [], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par zone

In [274]:
display(
    generate_box(
        children=generate_widget_all_classes(df_occupation_by_zone, ['occupation', 'zone_id', 'class_index'], [], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

### Par zone

#### Sans regroupements

In [243]:
display(
    generate_box(
        children=generate_widget_all_zones(df_detections_t, ['datetime', 'hour', 'T', 'class_index', 'zone_id'], ['datetime', 'T'], 450), 
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec durée par zone et jour

In [247]:
display(
    generate_box(
        children=generate_widget_all_zones(df_duration, ['date', 'duration', 'T', 'class_index', 'zone_id'], ['date', 'T'], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec durée par zone

In [270]:
display(
    generate_box(
        children=generate_widget_all_zones(df_duration_by_zone, ['duration', 'class_index', 'zone_id'], [], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par zone et par heure

In [248]:
display(
    generate_box(
        children=generate_widget_all_zones(df_occupation, ['date', 'hour', 'occupation', 'T', 'class_index', 'zone_id'], ['date', 'hour', 'T'], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Occupation par heure

In [279]:
display(
    generate_box(
        children=generate_widget_all_zones(df_occupation_by_hour, ['hour', 'occupation', 'zone_id', 'class_index'], [], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

### Par heure

#### Sans regroupement

In [252]:
display(
    generate_box(
        children=generate_widget_all_hours(df_detections_t, ['datetime', 'T', 'class_index', 'zone_id', 'hour'], ['datetime', 'T'], 360), 
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par zone et par heure

In [254]:
display(
    generate_box(
        children=generate_widget_all_hours(df_occupation, ['date', 'occupation', 'T', 'class_index', 'zone_id', 'hour'], ['date', 'T'], 360),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par heure

In [306]:
display(
    generate_box(
        children=generate_widget_all_hours(df_occupation_by_hour, ['hour', 'occupation', 'zone_id', 'class_index'], [], 360),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…